In [3]:
# ML-6: Tic-Tac-Toe using Q-Learning

import random
import numpy as np

In [4]:
# 1. Setting up the environment

wins = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
]

Q = {}
alpha = 0.1
gamma = 0.9
epsilon = 1.0
episodes = 50000

In [5]:
# 2. Defining the Tic-Tac-Toe game

def reset():
    return [0] * 9

def won(board, player):
    return any(all(board[i] == player for i in line) for line in wins)

def move(board, action, player):
    board[action] = player
    return board

def check_end(board):
    if won(board, 1):
        return 1
    if won(board, -1):
        return -1
    if 0 not in board:
        return 0
    return None

In [6]:
# 3. Building the reinforcement learning model

def act(state, actions, epsilon):
    if random.random() < epsilon:
        return random.choice(actions)

    values = [Q.get((state, a), 0) for a in actions]
    max_value = max(values)

    best_actions = [
        a for a in actions
        if Q.get((state, a), 0) == max_value
    ]

    return random.choice(best_actions)

def update_q(state, action, reward, next_state, next_actions):
    old_value = Q.get((state, action), 0)

    if next_actions:
        max_next = max(
            Q.get((next_state, a), 0)
            for a in next_actions
        )
    else:
        max_next = 0

    new_value = old_value + alpha * (
        reward + gamma * max_next - old_value
    )

    Q[(state, action)] = new_value

In [7]:
# 4. Training the model

for ep in range(episodes):

    board = reset()
    done = False

    while not done:

        state = tuple(board)
        actions = [i for i in range(9) if board[i] == 0]

        # Agent selects an action
        action = act(state, actions, epsilon)

        # Agent makes a move
        move(board, action, 1)

        result = check_end(board)

        if result is not None:
            reward = 1 if result == 1 else 0.5
            update_q(state, action, reward, None, [])
            break

        # Random opponent makes a move
        opponent_actions = [
            i for i in range(9) if board[i] == 0
        ]

        opponent_action = random.choice(opponent_actions)
        move(board, opponent_action, -1)

        result = check_end(board)

        if result is not None:
            reward = -1 if result == -1 else 0.5
            update_q(state, action, reward, None, [])
            break

        # Observe new state
        next_state = tuple(board)
        next_actions = [
            i for i in range(9) if board[i] == 0
        ]

        # Update Q-table
        update_q(state, action, 0, next_state, next_actions)

    # Gradually reduce exploration
    epsilon = max(0.01, np.exp(-0.0001 * ep))

In [8]:
# 5. Testing the model

w = d = l = 0

for _ in range(1000):

    board = reset()

    while True:

        state = tuple(board)
        actions = [i for i in range(9) if board[i] == 0]

        # Agent uses learned policy
        action = act(state, actions, 0)

        move(board, action, 1)

        result = check_end(board)

        if result is not None:
            if result == 1:
                w += 1
            else:
                d += 1
            break

        # Random opponent
        opponent_actions = [
            i for i in range(9) if board[i] == 0
        ]

        opponent_action = random.choice(opponent_actions)
        move(board, opponent_action, -1)

        result = check_end(board)

        if result is not None:
            if result == -1:
                l += 1
            else:
                d += 1
            break

# Display results

print("After Training:")
print("Total Episodes:", episodes)
print("Total Q-Table States:", len(Q))

print("\nTesting Results (1000 Games):")
print(f"Wins: {w} ({w/1000:.1%})")
print(f"Draws: {d} ({d/1000:.1%})")
print(f"Losses: {l} ({l/1000:.1%})")


After Training:
Total Episodes: 50000
Total Q-Table States: 6637

Testing Results (1000 Games):
Wins: 991 (99.1%)
Draws: 9 (0.9%)
Losses: 0 (0.0%)
